# GRU độc lập

Notebook này chỉ train GRU cho dự báo một phiên kế tiếp. Cấu hình và phép chia giống notebook tổng hợp: lookback 30, input copper/WTI/gold/silver, split theo thời gian 80/10/10, early stopping.

Metric được tính sau khi đổi dự đoán về giá gốc USD/pound. Chạy tuần tự từ trên xuống; notebook tự tìm CSV trong Kaggle Input hoặc thư mục data.

## 1. Nạp dữ liệu, cấu hình và chuẩn bị các hàm dùng chung

In [ ]:
from pathlib import Path
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler

SEEDS = [42]  # Tăng độ tin cậy bằng [42, 43, 44], đổi lại sẽ train lâu hơn.
LOOKBACK = 30
TRAIN_RATIO, VALIDATION_RATIO = 0.80, 0.10
EPOCHS, BATCH_SIZE, PATIENCE = 100, 32, 10
MULTISTEP_HORIZON = 5
TARGET = 'closed_copper_price'
FEATURES = [TARGET, 'close_wti_oil', 'close_gold', 'close_silver']
MODELS = ['SimpleRNN', 'LSTM', 'GRU', 'Bi-LSTM']
CSV_NAME = 'copper_investing_2006_2026.csv'

data_path = Path('data') / CSV_NAME
kaggle_input = Path('/kaggle/input')
if kaggle_input.exists():
    candidates = sorted(kaggle_input.rglob(CSV_NAME))
    if candidates:
        data_path = candidates[0]
if not data_path.exists():
    raise FileNotFoundError(
        f'Không tìm thấy {CSV_NAME}. Hãy attach CSV trong Kaggle Input '
        'hoặc đặt file trong data/.'
    )

df = pd.read_csv(data_path)
needed = {'date', *FEATURES}
missing = sorted(needed - set(df.columns))
if missing:
    raise ValueError(f'Dataset thiếu cột: {missing}')
df['date'] = pd.to_datetime(df['date'], errors='raise')
df = df.sort_values('date', kind='stable').reset_index(drop=True)
if df['date'].isna().any() or df['date'].duplicated().any():
    raise ValueError('Ngày bị rỗng hoặc trùng.')
df[FEATURES] = df[FEATURES].apply(pd.to_numeric, errors='raise')
if not np.isfinite(df[FEATURES].to_numpy(dtype=float)).all():
    raise ValueError('Các biến đầu vào có missing hoặc giá trị không hữu hạn.')

print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))
print('Dataset:', data_path, '| shape:', df.shape)
print('Date range:', df['date'].min().date(), '->', df['date'].max().date())


## 2. Chuẩn bị split, scaler và kiến trúc

In [ ]:
def prepare_data(frame, input_columns, target_column, lookback, horizon=1):
    n = len(frame)
    train_end = int(np.floor(n * TRAIN_RATIO))
    val_end = int(np.floor(n * (TRAIN_RATIO + VALIDATION_RATIO)))
    if not (0 < train_end < val_end < n):
        raise ValueError('Không đủ dữ liệu để chia train/validation/test.')

    x_raw = frame[input_columns].to_numpy(dtype=float)
    y_raw = frame[[target_column]].to_numpy(dtype=float)
    x_scaler, y_scaler = MinMaxScaler(), MinMaxScaler()
    x_scaler.fit(x_raw[:train_end])
    y_scaler.fit(y_raw[:train_end])
    x_scaled = x_scaler.transform(x_raw)

    def make_batch(start, end):
        first_origin = max(lookback - 1, start - 1)
        last_origin = min(end - horizon - 1, n - horizon - 1)
        origins = np.arange(first_origin, last_origin + 1)
        if len(origins) == 0:
            raise ValueError(f'Không đủ cửa sổ cho horizon={horizon}.')
        X = np.stack([
            x_scaled[o - lookback + 1:o + 1] for o in origins
        ]).astype('float32')
        indices = origins[:, None] + np.arange(1, horizon + 1)
        raw_y = y_raw[indices, 0]
        y = y_scaler.transform(raw_y.reshape(-1, 1))
        y = y.reshape(-1, horizon).astype('float32')
        return {'X': X, 'y': y, 'origins': origins, 'indices': indices}

    return {
        'train': make_batch(0, train_end),
        'val': make_batch(train_end, val_end),
        'test': make_batch(val_end, n),
        'x_scaler': x_scaler,
        'y_scaler': y_scaler,
        'train_end': train_end,
        'val_end': val_end,
    }


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)


def build_model(name, input_shape, output_steps=1):
    inputs = tf.keras.Input(shape=input_shape)
    if name == 'SimpleRNN':
        hidden = tf.keras.layers.SimpleRNN(32)(inputs)
    elif name == 'LSTM':
        hidden = tf.keras.layers.LSTM(32)(inputs)
    elif name == 'GRU':
        hidden = tf.keras.layers.GRU(32)(inputs)
    elif name == 'Bi-LSTM':
        hidden = tf.keras.layers.Bidirectional(
            tf.keras.layers.LSTM(32)
        )(inputs)
    else:
        raise ValueError(f'Model không hỗ trợ: {name}')
    outputs = tf.keras.layers.Dense(output_steps)(hidden)
    model = tf.keras.Model(inputs, outputs, name=name.replace('-', '_'))
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='mse',
    )
    return model


def fit_model(model, train, val):
    early_stop = tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', mode='min', patience=PATIENCE,
        restore_best_weights=True,
    )
    started = time.perf_counter()
    history = model.fit(
        train['X'], train['y'],
        validation_data=(val['X'], val['y']),
        epochs=EPOCHS, batch_size=BATCH_SIZE,
        shuffle=False, callbacks=[early_stop], verbose=0,
    )
    seconds = time.perf_counter() - started
    best_epoch = int(np.argmin(history.history['val_loss'])) + 1
    return history, seconds, best_epoch


def inverse_values(scaler, values):
    arr = np.asarray(values)
    return scaler.inverse_transform(arr.reshape(-1, 1)).reshape(arr.shape)


def metrics(actual, predicted):
    actual = np.asarray(actual, dtype=float).reshape(-1)
    predicted = np.asarray(predicted, dtype=float).reshape(-1)
    error = predicted - actual
    mse = float(np.mean(error ** 2))
    mape = (
        float(np.mean(np.abs(error / actual)) * 100)
        if np.all(np.abs(actual) > 1e-8) else np.nan
    )
    return {
        'mse': mse,
        'rmse': float(np.sqrt(mse)),
        'mae': float(np.mean(np.abs(error))),
        'mape_pct': mape,
    }


def output_folder():
    root = Path('/kaggle/working')
    if not root.exists():
        root = Path.cwd()
    folder = root / 'all_algorithm_comparison'
    folder.mkdir(parents=True, exist_ok=True)
    return folder


results_dir = output_folder()
print('Kết quả sẽ lưu tại:', results_dir)


## 3. Train và đánh giá GRU

Cell bên dưới chỉ khởi tạo model GRU. Kết quả nằm trong thư mục all_algorithm_comparison.

In [ ]:
MODEL_NAME = 'GRU'
seed = SEEDS[0]
one_step = prepare_data(df, FEATURES, TARGET, LOOKBACK, horizon=1)

tf.keras.backend.clear_session()
set_seed(seed)
model = build_model(MODEL_NAME, one_step['train']['X'].shape[1:])
history, training_seconds, best_epoch = fit_model(
    model, one_step['train'], one_step['val']
)

prediction_scaled = model(one_step['test']['X'], training=False).numpy()
prediction = inverse_values(one_step['y_scaler'], prediction_scaled).reshape(-1)
actual = inverse_values(
    one_step['y_scaler'], one_step['test']['y']
).reshape(-1)
test_metrics = metrics(actual, prediction)

result = pd.DataFrame([{
    'model': MODEL_NAME,
    'seed': seed,
    **test_metrics,
    'best_epoch': int(np.argmin(history.history['val_loss'])) + 1,
    'epochs_ran': len(history.epoch),
    'training_seconds': training_seconds,
    'parameters': int(model.count_params()),
}])
display(result.round(6))

model_folder = results_dir / MODEL_NAME.lower().replace('-', '')
model_folder.mkdir(parents=True, exist_ok=True)
result.to_csv(model_folder / 'metrics.csv', index=False)

test_dates = df.loc[one_step['test']['indices'][:, 0], 'date'].to_numpy()
prediction_frame = pd.DataFrame({
    'date': test_dates,
    'actual': actual,
    'prediction': prediction,
    'error': prediction - actual,
})
prediction_frame.to_csv(model_folder / 'predictions.csv', index=False)

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
axes[0].plot(history.history['loss'], label='Train')
axes[0].plot(history.history['val_loss'], label='Validation')
axes[0].set_title(f'{MODEL_NAME}: MSE loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE trên dữ liệu chuẩn hóa')
axes[0].grid(alpha=0.25)
axes[0].legend()

axes[1].plot(test_dates, actual, label='Giá thực tế', linewidth=1.7)
axes[1].plot(test_dates, prediction, label=f'Dự đoán {MODEL_NAME}')
axes[1].set_title(f'{MODEL_NAME}: dự báo một phiên trên tập test')
axes[1].set_xlabel('Ngày')
axes[1].set_ylabel('Giá đồng (USD/pound)')
axes[1].tick_params(axis='x', rotation=30)
axes[1].grid(alpha=0.25)
axes[1].legend()
plt.tight_layout()
plt.savefig(model_folder / 'forecast.png', dpi=160, bbox_inches='tight')
plt.show()

print('Đã lưu metric, dự đoán và biểu đồ tại:', model_folder)


RMSE/MAE/MSE/MAPE càng thấp càng tốt. Để so sánh trực tiếp GRU với Bi-LSTM, dùng cùng một dataset và protocol; có thể đối chiếu metric trong thư mục output hoặc chạy notebook tổng hợp Compare_All_Algorithms.ipynb.